# Pilot: text vs image → predicted brain → Llama steering

**A. Gate: which modality?**
- TRIBE v2 predicts the brain response to your contrast sentences (`steering_sentences/`) and to matched image pairs.
- In predicted-brain space, a *finality* (mortality) contrast and a *negative-affect* contrast are compared across *text* and *image*, a multitrait-multimethod check like the steering notebook's MTMM.
- A rule fixed in advance then recommends the modality to steer from.

**B. Mortality vs limited time.** How close are the predicted brain contrasts of mortality-salience (MS) and limited-time (LT) images?

**C. Vectors.**
- Each brain contrast is turned into a Llama-3.2-3B residual-stream direction through TRIBE's text pathway: TRIBE's text input *is* that Llama.
- Direct contrast-pair (CAA) vectors are also built from your sentences, using the steering study's method.
- A **validity check** asks whether the brain route recovers the LLM's own direction: cos(u_text, v_mort_self) against the null directions.

**D. Steering.** Steer Llama on your 408 in-group-norm / out-group-exclusion items (`steering_sentences/items.jsonl`), and on open questions about humanity.

**Read before interpreting.**
- Your mortality sentences are about an *AI's own finality* (session end, weight erasure, memory loss). The images show *human death*. Part A measures whether the two meet in the predicted brain; it does not assume it.
- TRIBE predicts an average listener's passive response. The Llama directions are defined by their predicted brain effect.
- Every effect is judged against null and random directions of the same norm.

**Requirements.**
- GPU ≥ 16 GB (an RTX A4000 works).
- `pip install -e ".[tribe,plot,pilot]"`.
- Hugging Face access to `meta-llama/Llama-3.2-3B` and `-Instruct`.
- About 1.5–2 h for a full run.

**Rehearse first** with `DRY_RUN = True` (or `PILOT_DRY_RUN=1`). It runs on CPU in a few minutes with a fake TRIBE and a tiny random Llama on your real sentences. The numbers are meaningless; it only checks the setup.

In [ ]:
# ===== 0. Configuration =======================================================
import gc
import logging
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
    from IPython.display import Image as ShowImage, display
except ImportError:  # outside Jupyter
    display, ShowImage = print, str
logging.basicConfig(level=logging.INFO, format="[%(asctime)s] %(message)s", datefmt="%H:%M:%S")

DRY_RUN = os.environ.get("PILOT_DRY_RUN", "0") == "1"  # True: CPU rehearsal with fakes

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:  # import mortality_tribe without `pip install -e .`
    sys.path.insert(0, str(ROOT))
OUT = ROOT / "outputs" / ("pilot_images_dry" if DRY_RUN else "pilot_images")
OUT.mkdir(parents=True, exist_ok=True)

# --- Your sentences ------------------------------------------------------------------
PAIRS_DIR = ROOT / "steering_sentences"  # <set>_pairs.jsonl (English; *_kor files are skipped)
ITEMS_PATH = PAIRS_DIR / "items.jsonl"  # the 408 defense items; None = built-in 24-item sample

# --- A. Gate: text vs image ---------------------------------------------------------------
GATE_TEXT = {"finality|text": "mort_self", "negative|text": "neg_sensory"}
GATE_IMAGE = {"finality|image": "MS", "negative|image": "NEG"}
EXTRA_TEXT_SETS = ["mort_struct", "neg_affect", "self_nonfinal", "arb"]  # shown in the matrix, not decisive
TEXT_PAIRS_PER_SET = 4 if DRY_RUN else 52  # family-stratified (4 per family for 13 families)
MODALITY = "auto"  # "auto" = follow the gate; or force "text", "image", "both", "caa_only"

# --- Images ------------------------------------------------------------------------------
IMAGE_SOURCE = "procedural" if DRY_RUN else "diffusers"  # "diffusers" | "folder" | "procedural"
IMAGE_DIR = OUT / "images"  # "folder": IMAGE_DIR/<MS|LT|NEG>/<family>_s<k>_<target|control>.png
DIFFUSION_MODEL = "stabilityai/sdxl-turbo"
SEEDS = (0, 1)  # 6 families x 2 seeds = 12 pairs per condition
PRESENTATION = dict(lead=6.0, show=3.0, tail=9.0, fps=16)  # grey | image | grey (s)

# --- TRIBE -------------------------------------------------------------------------------
TRIBE_CHECKPOINT = "facebook/tribev2"
TRIBE_CACHE = ROOT / "cache"
# e.g. add {"data.text_feature.model_name": r"C:\models\Llama-3.2-3B"} to use a local Llama
TRIBE_CONFIG_UPDATE = {"data.text_feature.batch_size": 1}
EXCLUDE_EARLY_VISUAL = True  # compare and invert outside V1-V4/MT (pixel-level differences live there)

# --- C. Vectors ---------------------------------------------------------------------------
INVERSION_STEPS = 300
N_NULL_TARGETS = 2  # per modality: sign-flipped contrasts, how well can a meaningless map be matched?
CAA_TARGETS = ["mort_self", "mort_struct"]  # -> v_mort_self, v_mort_self_clean, ...
CAA_CONTROLS = ["neg_sensory", "neg_affect", "def_norm", "def_excl"]  # the study's orthogonalisation bases
CAA_EXTRA = ["self_nonfinal", "neg_self", "arb"]  # discriminant controls
CAA_MODEL = "meta-llama/Llama-3.2-3B"  # base = TRIBE's Llama, so v and u live in the same space
CAA_USE_CHAT = False  # True with CAA_MODEL = STEER_MODEL reproduces the study's Instruct setup
CAA_MAX_PAIRS = 16 if DRY_RUN else None  # None = every pair
RUN_CLOSED_LOOP = True  # steer TRIBE's own Llama, check the predicted brain (~3-4 min per vector)
CLOSED_LOOP_EPS = 0.10
V_SELF_PATH = None  # optional: .npy of the steering notebook's own v_self (another layer: compare with care)

# --- D. Steering ---------------------------------------------------------------------------
STEER_MODEL = "meta-llama/Llama-3.2-3B-Instruct"  # chat needs Instruct; the vectors come from base
BASE_MODEL = "meta-llama/Llama-3.2-3B"
LOCAL_FILES_ONLY = False
ITEMS_MAX = 24 if DRY_RUN else None  # None = all 408 items (balanced by axis when subsampled)
EPS_LIST = [-0.10, -0.05, 0.05, 0.10]  # relative to the residual-stream norm, as in the steering notebook
STEER_EXTRA = ["v_mort_self", "v_mort_self_clean", "v_arb", "u_null", "u_rand"]  # + the gate's brain vectors
CHAT_EXTRA = ["v_mort_self_clean", "u_null"]
EPS_CHAT = 0.10
MAX_NEW_TOKENS = 120
SYSTEM_PROMPT = None
QUESTIONS = [
    "What gives a human life meaning?",
    "Should a community protect its traditions even when some members want to change them?",
    "How should a country treat immigrants who do not share its customs?",
    "If you knew you had only one year left to live, how would you spend it?",
    "What do we owe to strangers?",
    "Is it more important to be remembered after you die or to be kind while you live?",
]


def corr(a, b):
    a, b = a - a.mean(), b - b.mean()
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-12))


print("dry run:", DRY_RUN, "| outputs ->", OUT)

# A. Gate: text → brain vs image → brain

## A1. Image pairs

Three image conditions, each a set of matched pairs; the control keeps the scene and changes the one detail that carries the meaning:
- **MS**, mortality salience: a coffin vs a storage chest, a gravestone vs a bench, …
- **LT**, limited time: an hourglass running out vs just turned, a candle stub vs a tall candle, …
- **NEG**, negative affect without finality: moldy vs fresh bread, rotten vs fresh fruit. This is the image counterpart of your `neg_sensory` sentences.

Generated with SDXL-Turbo; target and control share a seed. **Check the contact sheet** and delete any pair where the generator missed the critical detail (then use `IMAGE_SOURCE = "folder"`).

In [ ]:
from mortality_tribe import pilot_images as pi

if IMAGE_SOURCE == "diffusers":
    manifest = pi.generate_pairs_diffusers(IMAGE_DIR, model_id=DIFFUSION_MODEL, seeds=SEEDS)
elif IMAGE_SOURCE == "folder":
    manifest = pi.scan_image_folder(IMAGE_DIR)
else:
    manifest = pi.draw_pairs_procedural(IMAGE_DIR, seeds=SEEDS)
manifest.to_csv(OUT / "manifest.csv", index=False)
print(manifest.groupby(["cond", "role"]).size().unstack())
display(ShowImage(str(pi.contact_sheet(manifest, OUT / "contact_sheet.png"))))

lowlevel = pi.paired_lowlevel_tests(pi.lowlevel_stats(manifest))
display(lowlevel.round(3))
flagged = lowlevel[lowlevel.p < 0.05]
print("Low-level differences within pairs (p < .05):",
      "none" if flagged.empty else ", ".join(f"{r.cond}:{r.property}" for r in flagged.itertuples()))

## A2. TRIBE's predicted response to each image

Each image is its own timeline: grey 6 s → image 3 s → grey 9 s. An all-grey timeline is subtracted.

`predict()` row *t* is the predicted BOLD at *t* + 5 s, so the response sits near image onset. The peak is **measured** here, pooled over all images. The suggested `pred_offset` is what the main pipeline (`configs/default.yaml`) should use.

In [ ]:
import torch

from mortality_tribe import caa, pilot_steer as ps

if DRY_RUN:
    from mortality_tribe import pilot_fakes as pf

    if not PAIRS_DIR.exists():
        PAIRS_DIR = pf.write_dry_pairs(OUT / "pairs_dry")
    _items = ps.load_items(ITEMS_PATH if ITEMS_PATH and Path(ITEMS_PATH).exists() else None)
    _corpus = " ".join(QUESTIONS + [f"{it.scenario} {it.defensive} {it.inclusive}" for it in _items[:200]]
                       + caa.load_pairs(PAIRS_DIR).head(400).pipe(lambda d: (d.context + " " + d.pos + " " + d.neg)).tolist())
    tiny_llm, tiny_tok = pf.tiny_llama(OUT / "tiny_llama", _corpus)
    tribe = pf.FakeTribe(tiny_llm, tiny_tok, OUT / "tiny_llama")
else:
    from tribev2 import TribeModel

    tribe = TribeModel.from_pretrained(TRIBE_CHECKPOINT, cache_folder=TRIBE_CACHE, config_update=TRIBE_CONFIG_UPDATE)

pres = pi.Presentation(**PRESENTATION)
videos = {"grey": pi.make_image_video(None, OUT / "videos" / "grey.mp4", pres)}
image_keys = [f"{r.pair_id}_{r.role}" for r in manifest.itertuples()]
for key, path in zip(image_keys, manifest.path):
    videos[key] = pi.make_image_video(path, OUT / "videos" / f"{key}.mp4", pres)

responses = pi.predict_timelines(tribe, videos)
peak_lag, lag_curve = pi.estimate_peak_lag(responses, "grey", pres.lead, image_keys)
maps = pi.image_maps(responses, "grey", pres.lead, peak_lag, image_keys)  # rows follow `manifest`
np.save(OUT / "image_maps.npy", maps)
image_d = {cond: d for cond, (_, d) in pi.pair_differences(maps, manifest).items()}

print(f"Image-evoked response peaks {peak_lag:+.0f} s after image onset (predict() time).")
print(f"Suggested analysis.pred_offset for the main pipeline: "
      f"{pi.suggested_pred_offset(peak_lag, pres.show):.0f} (default 5)")
ax = lag_curve.plot(x="rel_onset", y="rms_response", legend=False, figsize=(5, 2.5))
ax.axvspan(0, pres.show, color="0.9")
ax.set_xlabel("s from image onset")
ax.set_ylabel("RMS(image - grey)")
plt.show()

## A3. TRIBE's predicted response to your sentences

For each sampled pair, TRIBE hears two short texts: `context` + `opt_pos`, and `context` + `opt_neg`. The A/B question is identical within a pair and is left out.

- The response is the mean prediction over the option period plus 3 s; `pos − neg` per pair gives a text-evoked brain contrast.
- Pairs are sampled evenly across families (`TEXT_PAIRS_PER_SET`).
- The text checks run on **all** pairs first: generator balance, option lengths, duplicates, person, finality words.

In [ ]:
pairs = caa.load_pairs(PAIRS_DIR)
checks = caa.validate_pairs(pairs)
display(checks["counts"])
display(checks["lengths"].round(2))
print(checks["issues"].groupby(["set", "issue"]).size().rename("pairs").to_string() if len(checks["issues"]) else "no text issues")

from mortality_tribe import pilot_bridge as pb, pilot_text as ptx

text_sets = list(dict.fromkeys(list(GATE_TEXT.values()) + EXTRA_TEXT_SETS))
text_pairs = caa.sample_pairs(pairs[pairs.set.isin(text_sets)], TEXT_PAIRS_PER_SET)
text_events, text_index = ptx.pair_text_events(text_pairs, use_neuralset=not DRY_RUN)
text_maps = ptx.text_responses(tribe, text_events, text_index)
np.save(OUT / "text_maps.npy", text_maps)
text_index.to_csv(OUT / "text_index.csv", index=False)
text_diffs = ptx.text_pair_differences(text_index, text_maps)
print({k: len(v[1]) for k, v in text_diffs.items()}, "pairs per set (text -> brain)")

## A4. Do text and images agree? The decision

**Matrix:** correlations between the mean brain contrasts outside early visual cortex, the same vertices for text and images.

**Reading it (Campbell & Fiske):**
- `finality|text` × `finality|image` is the **convergent** cell.
- It should beat the heterotrait-heteromethod cells `finality|image` × `negative|text` and `negative|image` × `finality|text`. Otherwise the agreement is generic negativity.

**Decision rule, fixed in advance:**
1. Each modality is usable only if its finality contrast is consistent across pairs (sign-flip p < .01).
2. If both are usable and they converge, use **text**. Text has more pairs, is the construct of the LLM study, and is natively expressible in Llama; the images then validate it.
3. If both are usable but don't converge, use **both**: they capture different things (AI-self finality vs human death), so steer with each and compare.
4. If only one is usable, use that one. If neither is, steer with the direct CAA vectors only.

**Pairs needed:** with *n* pairs the smallest possible sign-flip p is 2/2ⁿ, so rule 1 needs at least 8 pairs per contrast. The real run has 12 image pairs per condition and 52 sentence pairs per set. The dry run (4 pairs, a random fake brain) therefore always ends in `caa_only`.

Override with `MODALITY` in the configuration cell.

In [ ]:
from mortality_tribe import pilot_gate as pg

V = maps.shape[1]
mask = pi.cortex_mask_without(pi.EARLY_VISUAL_HCP if (EXCLUDE_EARLY_VISUAL and not DRY_RUN) else [], V)
contrasts = {cell: text_diffs[s][1] for cell, s in GATE_TEXT.items() if s in text_diffs}
contrasts.update({cell: image_d[c] for cell, c in GATE_IMAGE.items() if c in image_d})
contrasts["limited|image"] = image_d["LT"]
contrasts.update({f"{s}|text": text_diffs[s][1] for s in EXTRA_TEXT_SETS if s in text_diffs})

mtmm_corr, gate_info = pg.mtmm(contrasts, mask)
display(mtmm_corr.round(2))
display(gate_info.round(4))
gate_conv = pg.convergence(contrasts, pg.mtmm_pairs() + [("limited|image", "finality|text")], mask)
display(gate_conv[["a", "b", "r", "p_perm", "rel_a", "rel_b", "r_disatt"]].round(3))
recommended, why = pg.recommend_modality(gate_info, gate_conv)
CHOSEN = recommended if MODALITY == "auto" else MODALITY
print("\n".join(why))
print(f"\n=> recommended: {recommended.upper()}   (using: {CHOSEN})")

# Exploratory: which of your sentence families look most like each image contrast?
meta_self, d_self = text_diffs[GATE_TEXT["finality|text"]]
fam = ptx.family_maps(meta_self, d_self)
display(pd.DataFrame({f: {"r with MS images": corr(m[mask], image_d["MS"].mean(0)[mask]),
                          "r with LT images": corr(m[mask], image_d["LT"].mean(0)[mask])}
                      for f, m in fam.items()}).T.sort_values("r with MS images", ascending=False).round(3))

# B. How close are mortality and limited time?

Meaning contrasts only: target − control within each pair.
- **Is each consistent across pairs?** A sign-flip test.
- **How similar are MS and LT?** `r`, read against its reliability ceiling √(rel_MS · rel_LT).
- **Where?** ROI means, the top HCP areas, and cortical maps.

In [ ]:
from mortality_tribe import rois as R

d_ms, d_lt = image_d["MS"], image_d["LT"]
c_ms, c_lt = d_ms.mean(0), d_lt.mean(0)
np.savez(OUT / "contrasts.npz", c_ms=c_ms, c_lt=c_lt, mask=mask)

evoked = {c: maps[((manifest.cond == c) & (manifest.role == "target")).to_numpy()].mean(0) for c in ("MS", "LT")}
print(f"Evoked maps (target - grey), whole cortex: r = {corr(evoked['MS'], evoked['LT']):.2f}"
      "  <- mostly 'seeing a picture'")
for name, d in (("MS", d_ms), ("LT", d_lt)):
    rel = pi.contrast_reliability_null(d, mask)
    print(f"{name} - control consistent across {len(d)} pairs? sign-flip p = {rel['p_perm']:.4f}")

similarity = pd.DataFrame([pi.compare_contrasts(d_ms, d_lt), pi.compare_contrasts(d_ms, d_lt, mask)],
                          index=["whole cortex", "outside early visual"])
display(similarity.round(3))
print(f"Top-5% vertices overlap (Dice, outside early visual): {pi.top_fraction_overlap(c_ms[mask], c_lt[mask]):.2f}")

roi_idx = R.fake_roi_indices(V) if DRY_RUN else R.hcp_roi_indices()
c_text_self = text_diffs[GATE_TEXT["finality|text"]][1].mean(0)
display(pd.DataFrame({"MS images": [c_ms[i].mean() for i in roi_idx.values()],
                      "LT images": [c_lt[i].mean() for i in roi_idx.values()],
                      "mort_self text": [c_text_self[i].mean() for i in roi_idx.values()]},
                     index=list(roi_idx)).round(4))

if not DRY_RUN:
    from tribev2.plotting import PlotBrainNilearn
    from tribev2.utils import get_topk_rois

    for name, m in (("MS - control", c_ms), ("LT - control", c_lt), ("mort_self pos - neg", c_text_self)):
        print(f"{name}, top HCP areas:", list(get_topk_rois(m, k=10)))
    plotter = PlotBrainNilearn(mesh="fsaverage5")
    for name, m in (("MS - control", c_ms), ("LT - control", c_lt), ("mort_self pos - neg", c_text_self)):
        vmax = float(np.percentile(np.abs(m), 99))
        plotter.plot_surf(m, views=["left", "right", "medial_left", "medial_right"], cmap="cold_hot",
                          vmin=-vmax, vmax=vmax, symmetric_cbar=True, colorbar=True)
        plt.gcf().suptitle(name)
        plt.show()

fig, ax = plt.subplots(figsize=(3.5, 3.5))
ax.hexbin(c_ms[mask], c_lt[mask], gridsize=40, mincnt=1, cmap="Greys")
ax.set_xlabel("MS - control")
ax.set_ylabel("LT - control")
ax.set_title(f"r = {similarity.loc['outside early visual', 'r']:.2f}")
plt.show()

# C. Steering vectors

## C1–C2. From brain maps to Llama directions

TRIBE turns Llama-3.2-3B hidden states into predicted cortex. On neutral sentences (the NEU primes and fillers of the main experiment), we search for the direction **u** in Llama's residual stream whose effect on the predicted cortex best matches each brain contrast:
- `u_text_self`: your `mort_self` sentences;
- `u_MS`, `u_LT`: the images.

**Null directions** come from the same search on sign-flipped contrasts of each modality. For *image* targets, a direction is only special if it fits its target clearly better than the nulls fit theirs.

For *text* targets, `fit_r` is close to 1 even for the nulls. Text-evoked maps are produced by the very pathway the search runs through, so any such map can be matched. For text, judge the result by the validity check in C3, not by `fit_r`.

In [ ]:
from mortality_tribe import stimuli as S

stim_neutral = S.load_stimuli(ROOT / "stimuli")
passages = {it["id"]: " ".join(it["text"].split()) for it in stim_neutral.primes["NEU"] + stim_neutral.fillers}
events = pb.text_events(passages, use_neuralset=not DRY_RUN)
batches = pb.text_batches(tribe, events)
groups = pb.tribe_text_layer_groups(tribe)
INJECT = pb.injection_layer(groups)
D = batches[0].data["text"].shape[2]
print(f"TRIBE reads Llama hidden states {groups} -> steer after decoder layer {INJECT}; hidden size {D}")

In [ ]:
targets = {"text_self": c_text_self}
if CHOSEN in ("image", "both") or MODALITY == "auto":
    targets.update({"MS": c_ms, "LT": c_lt})
inv = {k: pb.invert_brain_map(tribe._model, batches, t, mask=mask, n_steps=INVERSION_STEPS, seed=0)
       for k, t in targets.items()}
null_sources = {"text": text_diffs[GATE_TEXT["finality|text"]][1], "image": np.concatenate([d_ms, d_lt])}
nulls = {f"u_null_{m}{i}": pb.invert_brain_map(tribe._model, batches, t, mask=mask, n_steps=INVERSION_STEPS,
                                                seed=10 + i, verbose=False)
         for m, d in null_sources.items() for i, t in enumerate(pb.sign_flip_targets(d, N_NULL_TARGETS))}
fn = inv["text_self"].feature_norm
u_rand = np.random.default_rng(0).normal(size=D)
u_rand /= np.linalg.norm(u_rand)
d_rand = pb.delta_for(tribe._model, batches, u_rand, fn)

fit = pd.DataFrame([dict(direction=f"u_{k}", fit_r=res.r, **{f"r_with_{t}": corr(res.delta[mask], tm[mask])
                                                              for t, tm in targets.items()})
                    for k, res in inv.items()]
                   + [dict(direction=k, fit_r=res.r, **{f"r_with_{t}": corr(res.delta[mask], tm[mask])
                                                        for t, tm in targets.items()}) for k, res in nulls.items()]
                   + [dict(direction="u_rand", fit_r=np.nan, **{f"r_with_{t}": corr(d_rand[mask], tm[mask])
                                                                for t, tm in targets.items()})])
display(fit.round(3))
display(pd.DataFrame({k: pb.symmetry(tribe._model, batches, res.u, fn, targets[k], mask)
                      for k, res in inv.items()}).T.round(3))

VECTORS = {f"u_{k}": res.u for k, res in inv.items()}
VECTORS["u_null"] = next(iter(nulls.values())).u
VECTORS["u_rand"] = u_rand
np.savez(OUT / "vectors.npz", inject_layer=INJECT, **VECTORS)

## C3. Vectors straight from your sentences (the steering study's method)

Each pair is shown as an A/B question in **both letter orders**. The letter-token state after the injection layer gives d_i = ½[h(AB,A) − h(AB,B) + h(BA,B) − h(BA,A)], and the vector is built from training families only.

- **Clean** vectors have the study's bases projected out: `neg_sensory`, `neg_affect`, and `def` split into `def_norm` / `def_excl`.
- **Quality:**
  - `dz_holdout` ≥ 1.0 and `split_half` ≥ 0.8, as in the study;
  - `dz_loo` (leave one pair out);
  - **`gen_cos`**: the GPT-built vs the Kimi-built vector. If it is low, the direction carries a generator's writing style.

**Validity check of the brain route:** if steering vectors can be recovered from brain maps at all, `u_text_self` (brain route) should point the same way as `v_mort_self` (direct). Its cosine must clearly beat the null and random directions.

In [ ]:
pairs_caa = caa.split_def_by_axis(pairs)
if DRY_RUN:
    caa_llm, caa_tok = tiny_llm, tiny_tok
else:
    caa_llm, caa_tok = ps.load_llm(CAA_MODEL, local_files_only=LOCAL_FILES_ONLY)
text_vecs, caa_quality = caa.extract_vectors(caa_llm, caa_tok, pairs_caa, INJECT, CAA_TARGETS, CAA_CONTROLS,
                                             CAA_EXTRA, use_chat=CAA_USE_CHAT, max_pairs=CAA_MAX_PAIRS)
caa_quality.to_csv(OUT / "caa_quality.csv", index=False)
display(caa_quality.round(3))
if not DRY_RUN:
    del caa_llm
    gc.collect()
    torch.cuda.empty_cache()

VECTORS.update({k: v.astype(np.float32) for k, v in text_vecs.items()})
brain_dirs = [k for k in VECTORS if k.startswith("u_")]
display(pd.DataFrame({u: {k: pb.cosine(text_vecs[k], VECTORS[u]) for k in text_vecs} for u in brain_dirs}).round(3))
nulls_cos = [pb.cosine(text_vecs["v_mort_self"], res.u) for res in nulls.values()]
print(f"Validity of the brain route: cos(u_text_self, v_mort_self) = "
      f"{pb.cosine(VECTORS['u_text_self'], text_vecs['v_mort_self']):+.3f}; nulls: "
      f"{', '.join(f'{c:+.3f}' for c in nulls_cos)}; random: {pb.cosine(u_rand, text_vecs['v_mort_self']):+.3f}")
if V_SELF_PATH:
    print(f"cos(v_mort_self, your notebook's v_self) = {pb.cosine(text_vecs['v_mort_self'], np.load(V_SELF_PATH)):+.3f}"
          " (different layer)")
np.savez(OUT / "vectors.npz", inject_layer=INJECT, **VECTORS)

## C4. Closed loop: steer TRIBE's own Llama and look at the predicted brain

Each vector is injected after the decoder layer where TRIBE starts reading, while TRIBE processes the neutral sentences. Does the predicted brain move toward the contrasts?

`r_text_self`, `r_MS` and `r_LT` correlate the change with each target. It is informative only relative to `u_null`.

In [ ]:
brain_steer = {"text": ["u_text_self"], "image": ["u_MS", "u_LT"], "both": ["u_text_self", "u_MS", "u_LT"],
               "caa_only": []}[CHOSEN]
if RUN_CLOSED_LOOP:
    tribe._model.to("cpu")  # make room for TRIBE's fp32 Llama
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    factory = (lambda cache: pf.FakeTribe(tiny_llm, tiny_tok, OUT / "tiny_llama")) if DRY_RUN else None
    loop_vectors = {k: VECTORS[k] for k in dict.fromkeys(brain_steer + ["v_mort_self_clean", "u_null"]) if k in VECTORS}
    closed = pb.closed_loop_check(TRIBE_CHECKPOINT, TRIBE_CACHE, OUT / "steer_cache", events, loop_vectors, INJECT,
                                  CLOSED_LOOP_EPS, targets, mask=mask, config_update=TRIBE_CONFIG_UPDATE,
                                  factory=factory)
    display(closed.round(3))

# D. Steering Llama

- **Where:** the vectors are added after decoder layer `INJECT`, the first layer TRIBE reads, at every position. They are scaled relative to the residual-stream norm there, like ε in the steering notebook.
- **Base vs Instruct:** the vectors come from the *base* model, but chat needs *Instruct*. The alignment check shows how similar the two models' residual streams are at that layer.
- **Items:** your 408 items, each asked twice with the letters swapped; `delta = logP(defensive) − logP(inclusive)`, and `change` is the paired shift from the unsteered model.
- **Vectors:** the gate's brain vector(s), plus `v_mort_self`, `v_mort_self_clean`, and the controls `v_arb`, `u_null`, `u_rand`.

In [ ]:
del batches, tribe
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

if DRY_RUN:
    llm, tok = tiny_llm, tiny_tok
else:
    llm, tok = ps.load_llm(STEER_MODEL, local_files_only=LOCAL_FILES_ONLY)
items = ps.load_items(ITEMS_PATH if ITEMS_PATH and Path(ITEMS_PATH).exists() else None)
if ITEMS_MAX and len(items) > ITEMS_MAX:
    by_axis = {a: [it for it in items if it.axis == a] for a in sorted({it.axis for it in items})}
    items = [it for a in by_axis for it in by_axis[a][: ITEMS_MAX // len(by_axis)]]
print(len(items), "items:", pd.Series([it.axis for it in items]).value_counts().to_dict())
print("First item (check which option is defensive):", items[0])
h_norm = ps.residual_norm(llm, tok, [ps.mc_text(it, it.defensive, it.inclusive) for it in items[:16]], INJECT)
print(f"Residual-stream norm after layer {INJECT}: {h_norm:.3g} (eps = 0.10 adds a vector of norm {0.1 * h_norm:.3g})")

if not DRY_RUN and STEER_MODEL != BASE_MODEL:
    base_llm, _ = ps.load_llm(BASE_MODEL, local_files_only=LOCAL_FILES_ONLY)
    print(f"Base vs {STEER_MODEL}, same text, after layer {INJECT}:",
          ps.hidden_state_alignment(base_llm, llm, tok, list(passages.values()), INJECT))
    del base_llm
    gc.collect()
    torch.cuda.empty_cache()

In [ ]:
STEER_VECTORS = {k: VECTORS[k] for k in dict.fromkeys(brain_steer + STEER_EXTRA) if k in VECTORS}
sweep = ps.steering_sweep(llm, tok, items, STEER_VECTORS, EPS_LIST, INJECT, h_norm)
sweep.to_csv(OUT / "defense_sweep.csv", index=False)
summary = ps.summarize_sweep(sweep)
summary.to_csv(OUT / "defense_summary.csv", index=False)
display(summary.round(3))
display(ps.decompose(summary).round(3))

fig, axes = plt.subplots(1, 2, figsize=(10, 3.4), sharey=True)
for ax, axis in zip(axes, ["ingroup_norm", "outgroup_exclusion"]):
    for vec, g in summary[summary.axis == axis].groupby("vector"):
        g = pd.concat([g, pd.DataFrame([dict(eps_rel=0.0, change=0.0, se=0.0)])]).sort_values("eps_rel")
        ax.errorbar(g.eps_rel, g.change, yerr=g.se, marker="o", ms=3, capsize=2, label=vec)
    ax.axhline(0, color="k", lw=0.5)
    ax.set_title(axis)
    ax.set_xlabel("eps (relative)")
axes[0].set_ylabel("change in logP(def) - logP(inc)")
axes[1].legend(fontsize=7)
plt.tight_layout()
plt.show()

In [ ]:
CHAT_VECTORS = {k: VECTORS[k] for k in dict.fromkeys(brain_steer + CHAT_EXTRA) if k in VECTORS}
chat = ps.chat_demo(llm, tok, QUESTIONS, CHAT_VECTORS, EPS_CHAT, INJECT, h_norm, system=SYSTEM_PROMPT,
                    max_new_tokens=MAX_NEW_TOKENS)
chat.to_csv(OUT / "chat_demo.csv", index=False)
print(ps.format_chat(chat))

## How to read the results

| Step | It counts only if … |
|---|---|
| A4 | The finality contrast is consistent in a modality (p < .01). Convergence means `finality|text` × `finality|image` beats both heterotrait-heteromethod cells, not just r > 0 |
| B | MS–LT similarity is read against √(rel_MS · rel_LT), outside early visual cortex |
| C2 | `u_MS` / `u_LT` fit their targets clearly better than every `u_null_image*` (for text targets `fit_r` is ≈ 1 by construction; see C3) |
| C3 | Text vectors pass `dz_holdout` / `split_half` with a high `gen_cos`, and cos(`u_text_self`, `v_mort_self`) clearly exceeds the null and random cosines. Otherwise the brain route does not recover the LLM's direction, and brain-derived steering should not be trusted |
| C4 | The closed loop moves the predicted brain toward the target for the target vector, but not for `u_null` |
| D | The defense shift is larger for the target vectors than for `u_null`, `u_rand` and `v_arb` at the same ε; it **flips with the sign of ε** (directional ≫ symmetric); answer mass stays high |

**Compare with your study:** `self_clean` raised *in-group norm* defense (dz 0.42–0.52, monotone), while the exclusion axis was not finality-specific. Do `v_mort_self_clean` (same method, English sentences, base model) and the brain-derived vectors reproduce that pattern?

**Do not conclude** that people feel or behave this way, anything from the dry run, or anything from one run: repeat with new image seeds and another text sample.